# Recover, Retarget, Guide — quick start on Colab

Runs Module 3 (the render-guided sampler) on the bundled example, `000000000265.2.005`, with a frozen Wan 2.1 I2V 14B: once as the backbone without guidance (arm A0) and once as ours (arm A3), then grades both against the reference and shows the videos.

**Runtime:** an A100 (40 GB) GPU runtime; the sampler loads the 14B transformer in fp8. Each run of 33 frames takes a few minutes. Google Drive is optional: mount it only if you want the outputs to outlive the session.

**You supply one file:** the source clip `000000000265.2.005.mp4` (a MiraData clip; it is not redistributed here). Everything else comes from this repository and its release asset.

In [ ]:
#@title 1 · Settings
REPO_URL = "https://github.com/jnheinrich451-eng/Recover-Retarget-Guide.git"  #@param {type:"string"}
ASSET_URL = "https://github.com/jnheinrich451-eng/Recover-Retarget-Guide/releases/download/v1.0/m3_example_265.2.005.zip"  #@param {type:"string"}
PAIR = "000000000265.2.005/human"  #@param ["000000000265.2.005/human", "000000000265.2.005/vehicle"]
FRAMES = 33  #@param {type:"integer"}
USE_DRIVE = False  #@param {type:"boolean"}
DATA_ROOT = "/content/DATA"  #@param {type:"string"}
SOURCE_CLIP = ""  #@param {type:"string"}
# DATA_ROOT: the folder that will hold cache/, reports/ and videos/ (with USE_DRIVE, a path under /content/drive/MyDrive keeps it).
# SOURCE_CLIP: a path to 000000000265.2.005.mp4 (on Drive or already uploaded); leave empty to be asked for an upload in step 4.
import os, subprocess, sys
from pathlib import Path
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    if not DATA_ROOT.startswith("/content/drive/"):
        DATA_ROOT = "/content/drive/MyDrive/rrg_data"
DATA = Path(DATA_ROOT); DATA.mkdir(parents=True, exist_ok=True)
os.environ["MT_ROOT"] = str(DATA); os.environ["MT_DATA"] = str(DATA)
print("data root:", DATA)

In [ ]:
#@title 2 · The repository and the GPU
REPO_DIR = Path("/content/repo")
if not (REPO_DIR / ".git").exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=False)
os.chdir(REPO_DIR)
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
print("repo:", REPO_DIR, "| gpu:", gpu or "none")
mem_mb = int(gpu.split(",")[-1].split()[0]) if gpu else 0
if mem_mb < 38000:
    print("WARNING: the sampler needs about 40 GB of GPU memory (fp8 weights); pick an A100 runtime.")

In [ ]:
#@title 3 · The Module 3 environment (diffusers, transformers, MoGe; a few minutes)
# No Drive and no tokens are needed here: the script reads secrets only when a Drive with them is mounted,
# and SKIP_VIDEOS=1 skips the author's private reference dataset (you supply the clip in step 4).
r = subprocess.run(["bash", "scripts/bootstrap_phase2.sh"], env={**os.environ, "SKIP_VIDEOS": "1"})
assert r.returncode == 0, "bootstrap failed; read the log above"

In [ ]:
#@title 4 · The bundled example (release asset) and the source clip
import shutil, urllib.request, zipfile
clip = PAIR.split("/")[0]
if not (DATA / "cache" / "m3" / clip).exists():
    z = Path("/content/m3_example.zip")
    print("downloading", ASSET_URL)
    urllib.request.urlretrieve(ASSET_URL, z)
    with zipfile.ZipFile(z) as zf:
        zf.extractall(DATA)
    print("unpacked into", DATA)
else:
    print("caches already present under", DATA / "cache")
vid_dir = DATA / "videos"; vid_dir.mkdir(exist_ok=True)
target = vid_dir / f"{clip}.mp4"
if not target.exists():
    if SOURCE_CLIP and Path(SOURCE_CLIP).exists():
        shutil.copyfile(SOURCE_CLIP, target)
    else:
        from google.colab import files
        print(f"upload {clip}.mp4")
        up = files.upload()
        name = next(iter(up))
        target.write_bytes(up[name])
print("source clip:", target, f"{target.stat().st_size / 1e6:.1f} MB")
for rel in [f"cache/m2_e/{PAIR}/handoff.json", f"cache/m2_d/{PAIR}/anchor.png", f"cache/m3/{PAIR}/lift.npz", f"cache/m3/{PAIR}/geometry.npz"]:
    assert (DATA / rel).exists(), f"missing {rel}: the asset did not unpack as expected"
print("caches of", PAIR, "present")

In [ ]:
#@title 5 · Sample: the backbone (A0) and ours (A3)
common = [sys.executable, "scripts/run_m3_s4_guided.py", "--drive-root", str(DATA), "--pair", PAIR, "--seed", "0", "--frames", str(FRAMES)]
ours = ["--arm", "A3", "--soft-mask", "--sky-mask"] + (["--object", "--obj-window", "0.85", "0.75"] if PAIR.endswith("/vehicle") else [])
for arm in (["--arm", "A0"], ours):
    print("\n$", " ".join(common[1:] + arm), flush=True)
    r = subprocess.run(common + arm)
    assert r.returncode == 0, "the sampler stopped; read the log above"
out_dir = DATA / "cache" / "m3" / PAIR
A0_NAME = f"A0_seed0_rec_F{FRAMES}.mp4"
A3_NAME = f"A3_seed0_rec_F{FRAMES}" + ("_obj_ow0.75" if PAIR.endswith("/vehicle") else "") + "_sm_sky.mp4"
for n in (A0_NAME, A3_NAME):
    print(n, "->", "ok" if (out_dir / n).exists() else "MISSING")

In [ ]:
#@title 6 · Grade both videos against the reference (CoTracker3 on the GPU)
r = subprocess.run([sys.executable, "scripts/measure_m3_a0.py", "--drive-root", str(DATA), "--videos-dir", str(DATA / "videos"),
                    "--pairs", PAIR, "--source-frames", str(FRAMES), "--videos", A0_NAME, A3_NAME, "source"])
rep = DATA / "reports" / "m3_s4" / "a0_motion.md"
print(rep.read_text(encoding="utf-8") if rep.exists() else "no report written; read the log above")

In [ ]:
#@title 7 · Watch them
import base64
from IPython.display import HTML, display
def tag(p, label):
    b = base64.b64encode(Path(p).read_bytes()).decode()
    return f"<div style='display:inline-block;margin:6px;text-align:center'><video src='data:video/mp4;base64,{b}' width='416' autoplay loop muted playsinline></video><br>{label}</div>"
display(HTML(tag(out_dir / A0_NAME, "backbone, no guidance (A0)") + tag(out_dir / A3_NAME, "ours (A3)")))